# Coleta das ações: NVIDIA (NVDA) e Nasdaq-100 (QQQ)
Pregões de 19/10 a 18/11/2026, o mesmo recorte do `coleta_youtube.ipynb` (18/10 é domingo). Uma linha por pregão. A coluna `data_pregao` é a chave para juntar com o YouTube e com as notícias. Fonte: Yahoo Finance, pela biblioteca `yfinance` (`pip install yfinance`).

In [ ]:
%pip install yfinance

In [ ]:
import os
import json
import pathlib
import numpy as np
import pandas as pd
import yfinance as yf

# Mesmo recorte do coleta_youtube.ipynb: pregões de 19/10 a 18/11/2026.
INICIO, FIM = "2026-10-19", "2026-11-18"
BUFFER_DIAS = 400                          # dias de calendário antes do INICIO: variação do 1º pregão + histórico para estimar o beta
JANELA_ESTIMACAO = 250                     # nº de pregões (antes do estudo) usados para estimar alfa e beta da NVDA contra o QQQ
FIM_ESTIMACAO = None                       # último dia da estimação (AAAA-MM-DD). None = dia anterior ao INICIO. Se o caso começou antes de 19/10, use a data anterior ao começo
TICKERS = {"nvda": "NVDA", "qqq": "QQQ"}   # prefixo das colunas -> ticker (QQQ = Nasdaq-100, para descontar o mercado)
FERIADOS_BOLSA = []                        # feriados da NYSE (AAAA-MM-DD); nenhum entre 19/10 e 18/11/2026. Igual ao do YouTube.
ARQUIVO_MANUAL = {}                        # plano B: nomes de arquivos dentro de data/raw, ex.: {"nvda": "NVDA.csv", "qqq": "QQQ.csv"}

def raiz_do_projeto():
    # Acha a raiz do repositório (a pasta com .git), de onde quer que o notebook rode, em qualquer computador.
    candidatos = []
    if os.environ.get("PROJETO_DIR"):                 # opcional: força o caminho pela variável de ambiente PROJETO_DIR
        candidatos.append(pathlib.Path(os.environ["PROJETO_DIR"]))
    arq = globals().get("__vsc_ipynb_file__")         # caminho do próprio notebook, quando aberto no VS Code
    if arq:
        candidatos.append(pathlib.Path(arq).parent)
    candidatos.append(pathlib.Path.cwd())             # pasta de execução do kernel
    for c in candidatos:
        try:
            c = c.resolve()
        except OSError:
            continue
        for d in [c, *c.parents]:
            if (d / ".git").exists():
                return d
    raise RuntimeError("Não achei a raiz do projeto (pasta com .git). Abra o notebook de dentro do repositório "
                       f"(pasta notebooks/) ou defina PROJETO_DIR. Tentei: {[str(c) for c in candidatos]}")


RAIZ = raiz_do_projeto()
RAW, PROC = RAIZ / "data" / "raw", RAIZ / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print("Raiz do projeto:", RAIZ)

# Calendário esperado de pregões (o mesmo critério do notebook do YouTube).
PREGOES = pd.bdate_range(INICIO, FIM, freq="C", holidays=FERIADOS_BOLSA)
print(len(PREGOES), "pregões esperados:", PREGOES[0].date(), "->", PREGOES[-1].date())

## Coleta
Baixa NVDA e QQQ de 400 dias antes do início até 18/11. O `end` do yfinance não é incluído, então o código soma um dia. O histórico extra serve para calcular a variação do primeiro pregão e para estimar o beta da NVDA contra o QQQ fora da janela do estudo. Os preços são ajustados (`auto_adjust=True`) e os dados brutos ficam em `data/raw/acoes_bruto_*.csv`.

**Plano B:** o yfinance não é uma API oficial e o Yahoo muda o site de vez em quando. Se a coleta falhar, baixe o CSV de cada ticker (Yahoo Finance, aba *Historical Data*, de meados de setembro de 2025 a 18/11/2026), salve em `data/raw/` e coloque o nome do arquivo em `ARQUIVO_MANUAL`. O CSV precisa ter as colunas `Date`, `Open`, `High`, `Low`, `Close` e `Volume`.

In [ ]:
def baixar(ticker):
    ini = (pd.Timestamp(INICIO) - pd.Timedelta(days=BUFFER_DIAS)).strftime("%Y-%m-%d")
    fim_exclusivo = (pd.Timestamp(FIM) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    h = yf.Ticker(ticker).history(start=ini, end=fim_exclusivo, auto_adjust=True)
    if h.empty:
        raise RuntimeError(f"{ticker}: o Yahoo não devolveu dados. Tente de novo ou use o plano B (ARQUIVO_MANUAL).")
    h.index = pd.DatetimeIndex(h.index).tz_localize(None).normalize()      # só a data do pregão
    return h[["Open", "High", "Low", "Close", "Volume"]]


def ler_manual(caminho):
    m = pd.read_csv(caminho)
    m.columns = [c.strip().title() for c in m.columns]
    m["Date"] = pd.to_datetime(m["Date"])
    return m.set_index("Date").sort_index()[["Open", "High", "Low", "Close", "Volume"]]


bruto = {}
for pref, tk in TICKERS.items():
    if pref in ARQUIVO_MANUAL:
        bruto[pref] = ler_manual(RAW / ARQUIVO_MANUAL[pref])
        origem = f"arquivo manual {ARQUIVO_MANUAL[pref]}"
    else:
        bruto[pref] = baixar(tk)
        origem = "yfinance"
    bruto[pref].index.name = "data"
    bruto[pref].to_csv(RAW / f"acoes_bruto_{tk.lower()}.csv")
    h = bruto[pref]
    print(f"{tk}: {len(h)} linhas, {h.index.min().date()} -> {h.index.max().date()} ({origem})")

## Variações e retorno anormal
- `*_variacao_pct`: fechamento ÷ fechamento do pregão anterior − 1, em %.
- `nvda_gap_abertura_pct`: abertura ÷ fechamento anterior − 1, em %. Mostra a reação a notícias que saíram com a bolsa fechada (fim de semana ou depois das 16h), que é exatamente o que `data_pregao` desloca para o pregão seguinte.
- `retorno_anormal_pp`: variação da NVDA menos a do QQQ, em pontos percentuais. É uma correção simples, **sem beta**, mantida para comparação.
- `retorno_esperado_pct` e `retorno_anormal_beta_pp`: modelo de mercado. Estima-se `NVDA = alfa + beta × QQQ` por mínimos quadrados em até 250 pregões **anteriores** ao estudo (os dias do caso não entram na estimação); o retorno esperado é `alfa + beta × variação do QQQ` e o anormal é a variação observada menos o esperado. Use esta coluna como principal: a NVDA costuma oscilar mais que o QQQ (beta acima de 1), então a diferença simples superestima o "efeito próprio" nos dias de alta do mercado e subestima nos dias de queda. Os parâmetros ficam em `data/processed/acoes_modelo_mercado.json`.
- `nvda_volume`: ações negociadas no dia, que serve como medida de atenção do mercado.

As variações são calculadas antes do corte de período, então o primeiro pregão (19/10) usa o fechamento de 16/10. O código confere se os pregões que vieram do Yahoo são exatamente os do calendário esperado: se faltar ou sobrar algum dia, ele para, porque um calendário diferente do YouTube e das notícias estragaria o cruzamento.

In [ ]:
def preparar(h, pref, completo):
    h = h.sort_index()
    h = h[~h.index.duplicated(keep="last")]
    out = pd.DataFrame(index=h.index)
    out[f"{pref}_fechamento"] = h["Close"]
    out[f"{pref}_variacao_pct"] = h["Close"].pct_change() * 100
    if completo:
        out[f"{pref}_gap_abertura_pct"] = (h["Open"] / h["Close"].shift(1) - 1) * 100
        out[f"{pref}_volume"] = h["Volume"]
    return out


acoes = preparar(bruto["nvda"], "nvda", completo=True).join(preparar(bruto["qqq"], "qqq", completo=False), how="outer")
acoes["retorno_anormal_pp"] = acoes["nvda_variacao_pct"] - acoes["qqq_variacao_pct"]   # sem beta (para comparação)

# O calendário do Yahoo tem de ser igual ao esperado.
no_periodo = acoes.index[(acoes.index >= pd.Timestamp(INICIO)) & (acoes.index <= pd.Timestamp(FIM))]
faltando, sobrando = PREGOES.difference(no_periodo), no_periodo.difference(PREGOES)
if len(faltando) or len(sobrando):
    raise RuntimeError(f"Calendário diferente do esperado. Faltam: {[d.date().isoformat() for d in faltando]} | "
                       f"sobram: {[d.date().isoformat() for d in sobrando]}. Se for feriado da bolsa, ajuste FERIADOS_BOLSA "
                       "aqui e no notebook do YouTube.")

# Modelo de mercado: NVDA = alfa + beta × QQQ, estimado ANTES do estudo (os dias do caso não entram).
fim_est = pd.Timestamp(FIM_ESTIMACAO) if FIM_ESTIMACAO else PREGOES[0] - pd.Timedelta(days=1)
est = acoes.loc[acoes.index <= fim_est, ["nvda_variacao_pct", "qqq_variacao_pct"]].dropna().tail(JANELA_ESTIMACAO)
if len(est) < 120:
    raise RuntimeError(f"Só {len(est)} pregões para estimar o beta (mínimo 120). Aumente BUFFER_DIAS ou confira os dados do Yahoo.")
x, y = est["qqq_variacao_pct"], est["nvda_variacao_pct"]
beta = float(np.cov(x, y, ddof=1)[0, 1] / np.var(x, ddof=1))
alfa = float(y.mean() - beta * x.mean())
r2 = float(np.corrcoef(x, y)[0, 1] ** 2)
acoes["retorno_esperado_pct"] = alfa + beta * acoes["qqq_variacao_pct"]
acoes["retorno_anormal_beta_pp"] = acoes["nvda_variacao_pct"] - acoes["retorno_esperado_pct"]
modelo = {"beta": round(beta, 6), "alfa_pct_dia": round(alfa, 6), "r2": round(r2, 4), "n_obs": int(len(est)),
          "inicio_estimacao": est.index.min().date().isoformat(), "fim_estimacao": est.index.max().date().isoformat()}
(PROC / "acoes_modelo_mercado.json").write_text(json.dumps(modelo, ensure_ascii=False, indent=1), encoding="utf-8")
print("Modelo de mercado:", modelo)

acoes = acoes.loc[PREGOES]
acoes.index.name = "data_pregao"
for c in [c for c in acoes.columns if c.endswith(("_pct", "_pp"))]:
    acoes[c] = acoes[c].round(4)
acoes = acoes.reset_index()
acoes["data_pregao"] = acoes["data_pregao"].dt.date
acoes["data_coleta"] = pd.Timestamp.now(tz="America/New_York").date()

acoes.to_csv(PROC / "acoes_nvidia.csv", index=False)
print(len(acoes), "pregões salvos em", PROC / "acoes_nvidia.csv")

## Conferência
Confira o primeiro e o último pregão e os dias de maior queda e alta da NVDA. O `retorno_anormal_beta_pp` mostra quanto do movimento foi além do esperado pelo mercado (com beta); o `retorno_anormal_pp` é a versão simples, sem beta. Veja também os dias para conferir com uma segunda fonte.

In [ ]:
print(acoes[["data_pregao", "nvda_fechamento"]].iloc[[0, -1]].to_string(index=False))
print("\nValores ausentes:", int(acoes.drop(columns="data_coleta").isna().sum().sum()))

colunas = ["data_pregao", "nvda_variacao_pct", "qqq_variacao_pct", "retorno_anormal_pp", "retorno_anormal_beta_pp", "nvda_gap_abertura_pct", "nvda_volume"]
print("\n5 maiores quedas da NVDA:")
print(acoes.nsmallest(5, "nvda_variacao_pct")[colunas].to_string(index=False))
print("\n5 maiores altas da NVDA:")
print(acoes.nlargest(5, "nvda_variacao_pct")[colunas].to_string(index=False))
print("\nDias para conferir à mão com uma segunda fonte (|QQQ| >= 4% ou |NVDA| >= 10%):")
print(acoes.loc[(acoes["qqq_variacao_pct"].abs() >= 4) | (acoes["nvda_variacao_pct"].abs() >= 10), colunas].to_string(index=False))


In [ ]:
acoes

## Validação do CSV salvo
Lê o arquivo que será entregue e confere datas, valores ausentes, o cálculo das variações e o retorno anormal. Os três dias sorteados servem para conferir o fechamento à mão no Yahoo Finance (o preço é ajustado, então pode diferir um pouco do preço "cru" de outras fontes).

In [ ]:
salvo = pd.read_csv(PROC / "acoes_nvidia.csv", parse_dates=["data_pregao"])
numericas = salvo.drop(columns=["data_pregao", "data_coleta"])

print("pregões:", len(salvo), "| datas duplicadas:", int(salvo["data_pregao"].duplicated().sum()))
print("datas iguais ao calendário esperado:", list(salvo["data_pregao"]) == list(PREGOES))
print("valores ausentes:", int(numericas.isna().sum().sum()))
print("fechamentos <= 0:", int((salvo[["nvda_fechamento", "qqq_fechamento"]] <= 0).sum().sum()))
print("data_pregao sempre em dia útil:", bool((salvo["data_pregao"].dt.dayofweek < 5).all()))

for pref in ("nvda", "qqq"):   # a partir do 2º pregão, a variação salva tem de bater com os fechamentos salvos
    recalc = salvo[f"{pref}_fechamento"].pct_change() * 100
    ok = np.allclose(recalc.iloc[1:], salvo[f"{pref}_variacao_pct"].iloc[1:], atol=1e-3)
    print(f"{pref}: variação confere com os fechamentos:", ok)
print("retorno_anormal = NVDA - QQQ:", np.allclose(salvo["retorno_anormal_pp"], salvo["nvda_variacao_pct"] - salvo["qqq_variacao_pct"], atol=1e-3))
m = json.loads((PROC / "acoes_modelo_mercado.json").read_text(encoding="utf-8"))
esperado = m["alfa_pct_dia"] + m["beta"] * salvo["qqq_variacao_pct"]
print("retorno_anormal_beta = NVDA - (alfa + beta × QQQ):", np.allclose(salvo["retorno_anormal_beta_pp"], salvo["nvda_variacao_pct"] - esperado, atol=1e-3))
print(f"beta = {m['beta']} | R² = {m['r2']} | {m['n_obs']} pregões ({m['inicio_estimacao']} a {m['fim_estimacao']})")

for _, r in salvo.sample(3, random_state=1).iterrows():
    print(r["data_pregao"].date(), "| NVDA fechou em", round(r["nvda_fechamento"], 2), "| variação", r["nvda_variacao_pct"], "%")

## Teste de junção com o YouTube
Junta pela `data_pregao`. O YouTube tem um pregão a mais (19/11, com `cobertura_completa` falso), que sai na junção. Só roda se `youtube_diario_pregao.csv` existir.

In [ ]:
arq_yt = PROC / "youtube_diario_pregao.csv"
if arq_yt.exists():
    yt = pd.read_csv(arq_yt, parse_dates=["data_pregao"])
    j = salvo.merge(yt, on="data_pregao", how="inner", suffixes=("", "_yt"))
    print("pregões em comum:", len(j), "de", len(salvo), "das ações e", len(yt), "do YouTube")
    print("só no YouTube:", sorted(d.date().isoformat() for d in set(yt["data_pregao"]) - set(salvo["data_pregao"])))
    print("só nas ações:", sorted(d.date().isoformat() for d in set(salvo["data_pregao"]) - set(yt["data_pregao"])))
    print("pregões com cobertura completa do YouTube:", int(j["cobertura_completa"].sum()))
else:
    print("youtube_diario_pregao.csv ainda não existe: rode o coleta_youtube.ipynb primeiro.")

## Documentação da fonte e limitações (para o relatório)
**Fonte:** Yahoo Finance, acessado pela biblioteca `yfinance` na data da coluna `data_coleta`. Tickers: NVDA (NVIDIA) e QQQ (ETF do Nasdaq-100, usado como referência do mercado). Período: pregões de 19/10 a 18/11/2026, dias da bolsa de Nova York.

**Por que junto com as outras bases:** `data_pregao` é a mesma chave das tabelas do YouTube e das notícias, com a mesma regra de calendário: publicações de fim de semana ou depois das 16h (NY) caem no pregão seguinte.

**Limitações:**
1. O `yfinance` não é uma API oficial do Yahoo. Os valores podem diferir um pouco dos de outras fontes, e a coleta pode quebrar se o Yahoo mudar algo (por isso o plano B).
2. Os preços são ajustados por dividendos e desdobramentos (`auto_adjust=True`). Para o período do estudo isso muda pouco, mas o fechamento pode não coincidir com o "preço cru" de outros sites.
3. Os dados são diários. Não dá para separar o que aconteceu antes e depois de uma notícia dentro do mesmo pregão. `nvda_gap_abertura_pct` ajuda só com as notícias que saíram com a bolsa fechada.
4. `retorno_anormal_beta_pp` usa um modelo de mercado simples (alfa e beta estimados contra o QQQ em até 250 pregões anteriores ao estudo). Como a NVDA faz parte do QQQ, o beta pode ficar um pouco subestimado, e se o caso já tinha começado antes da janela de estimação o beta fica contaminado (ajuste `FIM_ESTIMACAO`). `retorno_anormal_pp` (sem beta) é mantido só para comparação. Outros fatos do mesmo dia (balanço de 17/11, dados econômicos, notícias da China, queda geral do mercado) também mexem na ação.
5. São cerca de 23 pregões, e a maior parte das variações grandes está em poucos dias. Isso limita o que a análise estatística pode afirmar: os resultados descrevem uma relação observada neste período, não causa.